# Modified GBM v2 calibration & Monte Carlo — period 7-day 1-minute

> **This is not a 2-year regime file.** The evaluation window is calendar week **2022-10-17 → 2022-10-21** (5 regular sessions: 17–21 Oct 2022; weekend skipped). Calibration lookback default = **1 hour** (60 trading minutes). Rolling default = **minutely**. Prices are stitched to **continuous RTH trading time** (overnight/weekend removed). §6 uses the **same listed-expiry sampling/LSM/RMSE logic as the 2-year files** (natural expiration, DTE 7–60), with unique minute-accurate quote times.

**Period file:** **2022-10-17 → 2022-10-21 (1-minute bars; 5 RTH sessions)**.

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Section roles**
- **§4 Calibration only:** lookback default is 1 hour; rolling default is minutely, **Reestimate**, inspect estimated parameters (no Monte Carlo plots here).
- **§5 Monte Carlo only:** Start / Restart; simulated paths and history comparison.
- **§6 Optimal stopping:** after §4 (and §5 paths), LSM on the systematic sample (Mondays / 15-min / 5-min; listed expiry, DTE 7–60), unique minute-accurate quote times; results in `stopping_results`.

True rolling rule: at each update, re-estimate the sign coins, the calm/wild coins, and the four lognormal \((\mu,\sigma)\) pairs from the current window and use them for the next MC segment. See `ROLLING_CALIBRATION.md`.




## 0. Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / ".." / "research" / "data"
TICKERS = ["AAPL", "MSFT", "SPY"]
BARS_PER_DAY = 390  # regular session 09:30–16:00 ET
N_DAYS = 252 * BARS_PER_DAY  # annualize 1-minute log returns
N_STEPS = 5000  # keep the full 1-minute grid (no daily-style subsample)
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

WINDOW_OPTIONS = {
    "1 hour": 60,   # trading minutes (not calendar hours)
    "1 day": 390,
}
ROLLING_OPTIONS = ["minutely", "hourly"]

WINDOW_ID = "2022-10-17_to_2022-10-21"
INTRADAY_DIR = DATA / "equity" / "short_interval" / "prices_1min_rth"
RATES_PATH = DATA / "rates" / "risk_free_dgs3mo_short_interval.csv"
OPT_DIR = DATA / "options" / "processed" / "short_interval"

def _with_option_days(fn, *args, **kwargs):
    """§6 uses a trading-day clock (N=252), same as the 2-year notebooks."""
    global N_DAYS
    saved = N_DAYS
    N_DAYS = 252
    try:
        return fn(*args, **kwargs)
    finally:
        N_DAYS = saved
_frames = []
for _t in TICKERS:
    _p = pd.read_csv(INTRADAY_DIR / f"{_t}.csv", parse_dates=["Datetime"]).set_index("Datetime").sort_index()
    _frames.append(_p["Close"].rename(_t))
GAP_MIN = pd.Timedelta(minutes=2)

def _session_gap(idx) -> pd.Series:
    return pd.Series(idx, index=idx).diff() > GAP_MIN

def stitch_continuous(close: pd.Series) -> pd.Series:
    """Rebuild a gapless trading-time price: overnight/weekend returns are not applied."""
    s = close.dropna()
    r = np.log(s).diff()
    r = r.mask(_session_gap(s.index), 0.0)
    r.iloc[0] = 0.0
    return pd.Series(float(s.iloc[0]) * np.exp(r.cumsum()), index=s.index, name=s.name)

def trading_x(idx) -> np.ndarray:
    return np.arange(len(idx))

def session_starts(idx) -> np.ndarray:
    g = _session_gap(idx).fillna(False).to_numpy()
    return np.flatnonzero(g)

def n_steps_to_expiry(quote_ts, expiry_ts=None) -> int:
    """Remaining 1-minute RTH bars from quote to expiration (PERIOD_END)."""
    q = pd.Timestamp(quote_ts)
    e = pd.Timestamp(expiry_ts) if expiry_ts is not None else PERIOD_END
    idx = prices.index
    n = int(((idx > q) & (idx <= e)).sum())
    return max(n, 2)
prices_raw = pd.concat(_frames, axis=1).sort_index()
PERIOD_START = pd.Timestamp("2022-10-17 09:30:00")
PERIOD_END = pd.Timestamp("2022-10-21 15:59:00")
prices = pd.concat(
    [stitch_continuous(prices_raw[t]).rename(t) for t in TICKERS],
    axis=1,
).sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()
# Do not treat the stitched 0-return at session joins as a 1-minute observation
for _t in TICKERS:
    _g = _session_gap(prices[_t].dropna().index)
    log_returns_all.loc[_g.reindex(log_returns_all.index, fill_value=False), _t] = np.nan


rolling = {}
cal_meta = {}

print(f"1-min sample: {prices.index.min()} → {prices.index.max()}")
print(
    f"Evaluation window: {len(period_prices)} 1-minute bars "
    f"({period_prices.index.min()} → {period_prices.index.max()}) — NOT a 2-year regime"
)
period_prices.head()

# --- clean plotting / widget memory (important after reopen) ---
plt.close("all")
plt.ioff()


## 1. Stock price trends (5-weekday 1-minute, continuous RTH (expiry 2022-10-21))

Overnight/weekend hours are dropped and prices are stitched into a continuous trading-time series. Dotted lines mark session joins. The return/vol panel uses 1-minute log returns on that same series.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    x = trading_x(s.index)
    ax.plot(x, s.values, color=COLORS[ticker], lw=1.4)
    for br in session_starts(s.index):
        ax.axvline(br, color="0.75", lw=0.8, ls=":")
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Close")
    ax.set_title(f"{ticker} ({role}) — continuous trading-time close (RTH only)")
axes[-1].set_xlabel("trading minute (overnight/weekend removed)")
fig.suptitle("Stock price trends — 5-weekday 1-minute, continuous RTH (expiry 2022-10-21)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))

# Realized volatility from 1-minute log returns (annualized with 252 × 390)
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
vol_rows = []
for ax, ticker in zip(axes, TICKERS):
    r = log_returns_all[ticker].loc[PERIOD_START:PERIOD_END].dropna()
    rv = r.rolling(30, min_periods=10).std() * np.sqrt(N_DAYS)
    ax.plot(trading_x(rv.index), rv.values, color=COLORS[ticker], lw=1.0)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("σ̂ (ann.)")
    ax.set_title(f"{ticker} ({role}) — 30-min rolling vol from 1-min returns")
    vol_rows.append({
        "ticker": ticker,
        "n_bars": int(r.shape[0]),
        "sigma_1min": float(r.std(ddof=1)),
        "sigma_ann": float(r.std(ddof=1) * np.sqrt(N_DAYS)),
        "mu_ann": float(r.mean() * N_DAYS),
    })
axes[-1].set_xlabel("trading minute")
fig.suptitle("Minute-level realized volatility — continuous RTH", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(pd.DataFrame(vol_rows).set_index("ticker").round(6))


## 2. Strike prices in this period

Systematic sample, same rule as every model: **every 15 minutes** in RTH (~130 observations; 5 sessions × 26). Nearest-ATM listed call, DTE 7–60, listed expiry. No random dates. Quote times are unique 1-minute stamps. RMSE is percentage RMSE vs market.


In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))
from american_lsm import load_calls, sample_listed_minute_calls

_section2 = {}
for ticker in TICKERS:
    df = sample_listed_minute_calls(
        load_calls(DATA, ticker, panel="short_interval"),
        prices[ticker],
        PERIOD_START,
        PERIOD_END,
        
    )
    _section2[ticker] = df
    uniq_t = df["trading_date"].nunique() if len(df) else 0
    n_exp = df["expiration"].dt.normalize().nunique() if len(df) else 0
    display(Markdown(
        f"### {ticker} — {len(df)} contracts, {uniq_t} unique 1-minute times, "
        f"{n_exp} listed expiries"
    ))
    if len(df):
        print("times unique:", bool(df["trading_date"].is_unique),
              "| minute grid:", bool((df["trading_date"].dt.floor("min") == df["trading_date"]).all()),
              "| expiries:", sorted({pd.Timestamp(x).normalize().date() for x in df["expiration"]}))
        cols = [c for c in ["trading_date", "S_t", "K", "expiration", "dte", "n_steps", "r", "moneyness", "option_price"] if c in df.columns]
        display(df[cols].round(4))
    else:
        print("No contracts.")


## 3. Estimation formulas (Modified GBM v2)

Same Markov direction as Modified GBM. Magnitudes are lognormal (always positive) and the
lognormal \((\mu,\sigma)\) pair is selected by a second 1-lag calm / wild chain.

**Stage 1 — direction.** Let \(U=\{R_t>0\}\) and \(D=\{R_t<0\}\). Laplace-smoothed

\[
\hat P(U\mid U),\ \hat P(D\mid U),\ \hat P(U\mid D),\ \hat P(D\mid D).
\]

**Stage 2 — calm / wild.** \(R_t=\ln(S_t/S_{t-1})\). A bar is wild (\(H\)) if \(|R|\) is above the
lookback median of \(|R|\), else calm (\(L\)). Laplace-smoothed

\[
\hat P(H\mid H),\ \hat P(L\mid H),\ \hat P(H\mid L),\ \hat P(L\mid L).
\]

**Stage 3 — size (Way B).** In each of the four buckets (up-calm, up-wild, down-calm, down-wild)
let \(m=\mathrm{mean}(|R|)\) and \(v=\mathrm{var}(|R|)\). Then

\[
\sigma^2=\log(1+v/m^2),\qquad \mu=\log(m)-\tfrac12\sigma^2,\qquad
\mathrm{size}=\exp\bigl(N(\mu,\sigma^2)\bigr).
\]

A bucket with fewer than two observations reuses the same-sign other regime, else the pooled same-sign moments.

**Stage 4 — price.** \(R_t=+\mathrm{size}\) on \(U\) and \(-\mathrm{size}\) on \(D\), then \(S_{t+1}=S_t e^{R_t}\).

**Stage 5 — \(Q\) (size-only).** Risk-neutral paths keep the \(P\)-measure sign coins and calm/wild coins.
If \(R^P=\pm\mathrm{size}\), choose one scale \(\lambda>0\) on that step so

\[
R^Q=\lambda R^P,\qquad \widehat{\mathbb{E}}[e^{R^Q}]=e^{r_f\Delta t}.
\]

Signs cannot flip. If no \(\lambda>0\) exists (a path cloud with no up move), fall back to the old additive shift.



## 4. Calibration only — 7-day 1-minute

Sliders + **Reestimate**. Shows **only** the rolling parameter graphs (no tables).  
Monte Carlo vs history is in **§5** only — one pair per company.




In [ ]:
def estimate_modified_gbm(log_rets: pd.Series):
    """Markov direction + lognormal sizes with a calm/wild Markov selector."""
    eps = 1e-8
    x = log_rets.dropna().astype(float)
    x = x[np.isfinite(x)]
    n = int(x.shape[0])
    nz = x[x != 0.0]
    if n < 3 or int(nz.shape[0]) < 3:
        return None
    signed = nz.to_numpy(dtype=float)
    mag = np.maximum(np.abs(signed), eps)
    up = signed > 0.0
    wild = mag > float(np.median(mag))

    prev_u, curr_u = up[:-1], up[1:]
    n_from_u = int(prev_u.sum())
    n_from_d = int((~prev_u).sum())
    n_uu = int((prev_u & curr_u).sum())
    n_dd = int((~prev_u & ~curr_u).sum())
    p_uu = (n_uu + 0.5) / (n_from_u + 1.0)
    p_dd = (n_dd + 0.5) / (n_from_d + 1.0)
    p_du = 1.0 - p_uu
    p_ud = 1.0 - p_dd

    prev_h, curr_h = wild[:-1], wild[1:]
    n_from_h = int(prev_h.sum())
    n_from_l = int((~prev_h).sum())
    n_hh = int((prev_h & curr_h).sum())
    n_ll = int((~prev_h & ~curr_h).sum())
    p_hh = (n_hh + 0.5) / (n_from_h + 1.0)
    p_ll = (n_ll + 0.5) / (n_from_l + 1.0)
    p_hl = 1.0 - p_hh
    p_lh = 1.0 - p_ll

    def _lognormal_params(arr):
        arr = np.asarray(arr, dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size < 2:
            return None
        m = float(np.mean(arr))
        if not np.isfinite(m) or m < eps:
            m = eps
        v = float(np.var(arr, ddof=1))
        if not np.isfinite(v) or v < 0.0:
            v = 0.0
        sig2 = float(np.log(1.0 + v / (m * m)))
        sig = float(np.sqrt(max(sig2, 0.0)))
        if sig <= 0.0:
            sig = 1e-12
        mu = float(np.log(m) - 0.5 * sig2)
        return mu, sig

    ul = _lognormal_params(mag[up & ~wild])
    uh = _lognormal_params(mag[up & wild])
    dl = _lognormal_params(mag[(~up) & ~wild])
    dh = _lognormal_params(mag[(~up) & wild])
    u_pool = _lognormal_params(mag[up])
    d_pool = _lognormal_params(mag[~up])
    all_pool = _lognormal_params(mag)

    def _fill(primary, same_sign, pooled):
        if primary is not None:
            return primary
        if same_sign is not None:
            return same_sign
        if pooled is not None:
            return pooled
        return (float(np.log(eps)), 1e-12)

    mu_u_l, sig_u_l = _fill(ul, uh, u_pool if u_pool is not None else all_pool)
    mu_u_h, sig_u_h = _fill(uh, ul, u_pool if u_pool is not None else all_pool)
    mu_d_l, sig_d_l = _fill(dl, dh, d_pool if d_pool is not None else all_pool)
    mu_d_h, sig_d_h = _fill(dh, dl, d_pool if d_pool is not None else all_pool)
    return {
        "n_days": n,
        "p_uu": float(p_uu),
        "p_du": float(p_du),
        "p_ud": float(p_ud),
        "p_dd": float(p_dd),
        "p_hh": float(p_hh),
        "p_ll": float(p_ll),
        "p_lh": float(p_lh),
        "p_hl": float(p_hl),
        "mu_u_l": float(mu_u_l),
        "sig_u_l": float(sig_u_l),
        "mu_u_h": float(mu_u_h),
        "sig_u_h": float(sig_u_h),
        "mu_d_l": float(mu_d_l),
        "sig_d_l": float(sig_d_l),
        "mu_d_h": float(mu_d_h),
        "sig_d_h": float(sig_d_h),
        "last_up": 1.0 if bool(up[-1]) else 0.0,
        "last_wild": 1.0 if bool(wild[-1]) else 0.0,
        "p_u": float(up.mean()),
        "p_h": float(wild.mean()),
    }



def _slice_window(rets: pd.Series, end: pd.Timestamp, n_bars) -> pd.Series:
    """Last n trading-time bars ending at `end` (calendar gaps already removed)."""
    sub = rets.loc[rets.index <= pd.Timestamp(end)]
    n = int(n_bars)
    return sub.iloc[-n:] if len(sub) else sub


def calibrate_ticker(ticker: str, window_label: str, rolling_mode: str) -> pd.DataFrame:
    rets = log_returns_all[ticker].dropna()
    offset = WINDOW_OPTIONS[window_label]
    rows = []

    period_idx = rets.loc[(rets.index >= PERIOD_START) & (rets.index <= PERIOD_END)].index
    if rolling_mode == "minutely":
        update_dates = period_idx
    elif rolling_mode == "hourly":
        hours = period_idx.floor("h")
        update_dates = pd.DatetimeIndex(
            [period_idx[hours == h].max() for h in hours.unique()]
        ).sort_values()
    else:
        update_dates = pd.DatetimeIndex([period_idx[0]])

    for t_u in update_dates:
        window = _slice_window(rets, pd.Timestamp(t_u), offset)
        est = estimate_modified_gbm(window)
        if est is None:
            continue
        rows.append({
            "date": pd.Timestamp(t_u),
            "window_start": window.index.min(),
            "window_end": window.index.max(),
            **est,
        })
    return pd.DataFrame(rows)


def _mc_time_grid(hist: pd.Series, n_steps: int = N_STEPS):
    """Evenly spaced 1-minute grid with exactly n_steps steps (n_steps+1 prices)."""
    hist = hist.dropna()
    n_full = len(hist)
    if n_full <= n_steps + 1:
        return hist
    idx = np.linspace(0, n_full - 1, n_steps + 1)
    idx = np.rint(idx).astype(int)
    for i in range(1, len(idx)):
        if idx[i] <= idx[i - 1]:
            idx[i] = min(idx[i - 1] + 1, n_full - 1)
    return hist.iloc[idx]

def param_schedule_for_steps(ticker: str, cal_table: pd.DataFrame):
    hist = _mc_time_grid(period_prices[ticker], N_STEPS)
    dates = hist.index
    n_steps = len(dates) - 1
    cal = cal_table.sort_values("date").reset_index(drop=True)
    cal_dates = pd.to_datetime(cal["date"]).to_numpy()
    cols = [
        "p_uu", "p_du", "p_ud", "p_dd", "p_hh", "p_ll", "p_lh", "p_hl",
        "mu_u_l", "sig_u_l", "mu_u_h", "sig_u_h",
        "mu_d_l", "sig_d_l", "mu_d_h", "sig_d_h",
        "last_up", "last_wild",
    ]
    arrs = {c: cal[c].to_numpy(dtype=float) for c in cols}
    steps = {c: np.empty(n_steps, dtype=float) for c in cols}
    for i in range(n_steps):
        idx = np.searchsorted(cal_dates, np.datetime64(dates[i]), side="right") - 1
        if idx < 0:
            idx = 0
        for c in cols:
            steps[c][i] = arrs[c][idx]
    return dates, steps, float(hist.iloc[0]), hist

def _show_fig(fig):
    """Show a figure exactly once as PNG.

    Root cause of duplicates: with %matplotlib inline, display(fig) inside an
    Output can ALSO be flushed again by the inline backend → same graph twice
    (worse after reopen when the old kernel is still alive). Saving PNG bytes
    and closing the Figure first avoids that second paint.
    """
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def plot_rolling_paths(rolling_dict: dict, window_label: str, rolling_mode: str):
    """Rolling Modified GBM v2 parameters (graphs only)."""
    panels = [
        (("p_uu", "p_dd"), r"$\hat P(U|U)$ / $\hat P(D|D)$", "Direction persistence"),
        (("p_hh", "p_ll"), r"$\hat P(H|H)$ / $\hat P(L|L)$", "Calm / wild persistence"),
        (("mu_u_l", "mu_u_h"), r"$\mu_{U,L}$ / $\mu_{U,H}$", "Up log-size means (calm / wild)"),
        (("mu_d_l", "mu_d_h"), r"$\mu_{D,L}$ / $\mu_{D,H}$", "Down log-size means (calm / wild)"),
    ]
    with plt.ioff():
        fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True)
        for ax, ((c1, c2), ylab, title) in zip(axes, panels):
            for t in TICKERS:
                r = rolling_dict[t]
                x = np.array([
                    period_prices.index.get_indexer([pd.Timestamp(d)], method="pad")[0]
                    for d in r["date"]
                ])
                mark = "o" if len(r) < 40 else None
                ax.plot(x, r[c1], lw=1.2, label=f"{t} {c1}", color=COLORS[t], marker=mark, ms=3)
                ax.plot(x, r[c2], lw=1.0, ls="--", color=COLORS[t], alpha=0.75, marker=mark, ms=3)
            ax.set_ylabel(ylab)
            ax.set_title(f"{title} — {rolling_mode}, lookback {window_label}")
            ax.legend(frameon=False, ncol=3, fontsize=8)
        axes[-1].set_xlabel("trading minute")
        fig.tight_layout()
    _show_fig(fig)

def reestimate(_=None):
    global rolling, cal_meta
    window_label = window_slider.value
    rolling_mode = rolling_slider.value
    rolling = {t: calibrate_ticker(t, window_label, rolling_mode) for t in TICKERS}
    cal_meta = {"window_label": window_label, "rolling_mode": rolling_mode}

    with cal_out:
        clear_output(wait=True)
        display(Markdown(
            f"**Calibration updated:** lookback=`{window_label}`, rolling=`{rolling_mode}` "
            f"(n_updates: " + ", ".join(f"{t}={len(rolling[t])}" for t in TICKERS) + ")"
        ))
        plot_rolling_paths(rolling, window_label, rolling_mode)
        display(Markdown("Go to **§5** and click **Start** for one MC pair per company."))


btn_reestimate.on_click(reestimate)
display(cal_ui)
reestimate()




## 5. Monte Carlo only — one graph pair per company (7-day 1-minute)

| Left | Right |
|------|--------|
| Monte Carlo paths + median | Median path + 25–75% band vs historical prices |

Uses latest **Reestimate** from §4. **Start** / **Restart** redraw that single pair (never stacks another copy).

**Stock-path metrics** (printed under each pair)

1. **MAE** — median absolute error of the 50th percentile path vs actual $S_t$ (central-tendency fit).
2. **ICP** — interval coverage probability: share of actual prices that fall inside the 25th–75th percentile band.
3. **Average band width** — mean($p_{75}-p_{25}$); how narrow or wide the model’s uncertainty range is.




In [ ]:
def _rn_size_scale(r_p, target):
    """Positive size scale λ with mean(exp(λ r_p)) = target. Keeps sign(r_p)."""
    r_p = np.asarray(r_p, dtype=float)
    target = float(target)
    if (not np.isfinite(target)) or target <= 0.0:
        return None
    log_t = float(np.log(target))
    if (not np.any(r_p > 0.0)) and log_t > 0.0:
        return None
    lam = 1.0
    for _ in range(40):
        x = lam * r_p
        m = float(np.max(x))
        e = np.exp(x - m)
        s = float(e.sum())
        lm = float(np.log(s / float(e.size)) + m)
        f = lm - log_t
        if abs(f) < 1e-13:
            return float(lam) if lam > 0.0 else None
        deriv = float(np.sum(r_p * e) / s)
        if (not np.isfinite(deriv)) or abs(deriv) < 1e-18:
            break
        lam = lam - f / deriv
        if (not np.isfinite(lam)) or lam <= 0.0:
            lam = 1e-8
    if lam > 0.0 and np.isfinite(lam):
        x = lam * r_p
        m = float(np.max(x))
        e = np.exp(x - m)
        lm = float(np.log(float(np.mean(e))) + m)
        if abs(lm - log_t) < 1e-10:
            return float(lam)
    return None


def simulate_modified_gbm_rolling(steps, S0, n_paths, seed, rf=None):
    """Modified GBM v2: Markov sign, Markov calm/wild, lognormal size, S * exp(r).

    `rf` is an annual risk-free rate. When set, sizes are scaled (not shifted)
    so E[exp(r_t)] = exp(rf / N_DAYS) and signs stay as drawn under P.
    Leave `rf=None` for P-measure paths.
    """
    rng = np.random.default_rng(seed)
    n_steps = len(steps["p_uu"])
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    up = np.full(n_paths, float(steps["last_up"][0]) >= 0.5, dtype=bool)
    wild = np.full(n_paths, float(steps["last_wild"][0]) >= 0.5, dtype=bool)
    rf_step = None if rf is None else float(rf) / float(N_DAYS)

    for i in range(n_steps):
        p_uu = float(np.clip(steps["p_uu"][i], 0.0, 1.0))
        p_ud = float(np.clip(steps["p_ud"][i], 0.0, 1.0))
        p_hh = float(np.clip(steps["p_hh"][i], 0.0, 1.0))
        p_ll = float(np.clip(steps["p_ll"][i], 0.0, 1.0))
        mu_u_l = float(steps["mu_u_l"][i])
        mu_u_h = float(steps["mu_u_h"][i])
        mu_d_l = float(steps["mu_d_l"][i])
        mu_d_h = float(steps["mu_d_h"][i])
        sig_u_l = max(float(steps["sig_u_l"][i]), 1e-12)
        sig_u_h = max(float(steps["sig_u_h"][i]), 1e-12)
        sig_d_l = max(float(steps["sig_d_l"][i]), 1e-12)
        sig_d_h = max(float(steps["sig_d_h"][i]), 1e-12)
        p_up = np.where(up, p_uu, p_ud)
        p_wild = np.where(wild, p_hh, 1.0 - p_ll)
        up = rng.random(n_paths) < p_up
        wild = rng.random(n_paths) < p_wild
        mu = np.where(up, np.where(wild, mu_u_h, mu_u_l), np.where(wild, mu_d_h, mu_d_l))
        sig = np.where(up, np.where(wild, sig_u_h, sig_u_l), np.where(wild, sig_d_h, sig_d_l))
        mag = np.exp(rng.normal(mu, sig))
        mag = np.maximum(mag, 1e-16)
        r_p = np.where(up, mag, -mag)
        if rf_step is None:
            r = r_p
        else:
            lam = _rn_size_scale(r_p, float(np.exp(rf_step)))
            if lam is None:
                mx = float(np.mean(np.exp(r_p)))
                r = r_p + (rf_step - np.log(max(mx, 1e-300)))
            else:
                r = lam * r_p
        paths[:, i + 1] = paths[:, i] * np.exp(r)
    return paths

def _show_fig(fig):
    """Show a figure exactly once as PNG.

    Root cause of duplicates: with %matplotlib inline, display(fig) inside an
    Output can ALSO be flushed again by the inline backend → same graph twice
    (worse after reopen when the old kernel is still alive). Saving PNG bytes
    and closing the Figure first avoids that second paint.
    """
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _draw_ticker_pair(ticker: str, out: widgets.Output, seed: int, n_paths: int = 1000):
    """Replace contents of `out` with exactly one 1×2 figure."""
    with out:
        clear_output(wait=True)
        if ticker not in rolling or len(rolling[ticker]) == 0:
            display(Markdown("Run **Reestimate** in §4 first."))
            return
        dates_now, steps_now, S0_now, hist_now = param_schedule_for_steps(
            ticker, rolling[ticker]
        )
        paths = simulate_modified_gbm_rolling(steps_now, S0_now, n_paths, seed)
        expected = paths.mean(axis=0)
        p25 = np.percentile(paths, 25, axis=0)
        p50 = np.percentile(paths, 50, axis=0)
        p75 = np.percentile(paths, 75, axis=0)
        _hist = np.asarray(hist_now.values, dtype=float)
        _n = min(len(p50), len(_hist))
        p25, p50, p75, expected, _hist = p25[:_n], p50[:_n], p75[:_n], expected[:_n], _hist[:_n]

        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
            tt = trading_x(dates_now)[:_n]
            axes[0].plot(tt, paths[:, :_n].T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(tt, p50, color="black", lw=2.0, ls="--", label="median path (50th)")
            axes[0].set_title(f"{ticker}: Monte Carlo")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].fill_between(tt, p25, p75, color=COLORS[ticker], alpha=0.18, lw=0, zorder=1, label="25–75% range")
            axes[1].plot(tt, _hist, color=COLORS[ticker], lw=1.8, label="historical", zorder=3)
            axes[1].plot(tt, p50, color="black", lw=2.0, ls="--", label="median path (50th)", zorder=4)
            axes[1].set_title(f"{ticker}: median vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)
            for ax in axes:
                ax.set_xlabel("trading minute")
            mae = float(np.mean(np.abs(p50 - _hist)))
            icp = float(np.mean((_hist >= p25) & (_hist <= p75)))
            abw = float(np.mean(p75 - p25))
            rmse = float(100.0 * np.sqrt(np.mean(((p50 - _hist) / np.maximum(np.abs(_hist), 1e-8)) ** 2)))
            fig.suptitle(
                f"{ticker} | MAE={mae:.4f} | ICP={100*icp:.1f}% | width={abw:.4f} | seed={seed} | "
                f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                fontsize=11,
                y=1.02,
            )
            fig.tight_layout()
        _show_fig(fig)
        display(Markdown(
            f"**MAE (50th vs $S_t$)** = `{mae:.4f}` · "
            f"**ICP (25–75)** = `{100*icp:.1f}%` · "
            f"**avg band width** = `{abw:.4f}` · "
            f"RMSE%(p50) = `{rmse:.2f}%` | seed = `{seed}`"
        ))


def make_ticker_panel(ticker: str, n_paths: int = 1000):
    """One Output per company. Start/Restart only replace that Output (no stacking)."""
    state = {"seed": 42}
    mode = cal_meta.get("rolling_mode", "?")
    win = cal_meta.get("window_label", "?")
    out = widgets.Output(layout=widgets.Layout(width="100%"))
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(f"<b>{ticker}</b> — one graph pair | lookback={win}, mode={mode}")

    busy = {"on": False}

    def on_start(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    def on_restart(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    btn_start.on_click(on_start)
    btn_restart.on_click(on_restart)
    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


plt.close("all")
plt.ioff()

mc_host = widgets.VBox([])
children = [widgets.HTML("<b>§5 Monte Carlo — click <i>Start</i> once per company (one pair only)</b>")]
for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    children.append(widgets.HTML(f"<h4 style='margin:8px 0 4px'>{ticker} ({role})</h4>"))
    children.append(make_ticker_panel(ticker))
mc_host.children = tuple(children)
display(mc_host)




## 6. Optimal stopping (American calls — Modified GBM v2)

Continuous with §5: after Monte Carlo stock paths are available, use the **same Modified GBM v2 simulator** and §4 calibration on **SPY / AAPL / MSFT** to decide exercise vs wait for American calls.

**Do not average simulated stock paths before stopping decisions.** Generate a cloud of individual risk-neutral paths (each keeps its own shocks). Then Longstaff–Schwartz:

1. At each exercise date, compute the immediate payoff $\max(S_t-K,0)$ **on every path**.
2. Estimate continuation by regression on in-the-money simulated states ($1, S, S^2$, path vol proxy).
3. **Each path** exercises iff payoff $>$ continuation; otherwise it continues.
4. Discount that path's stopping payoff to $t=0$.
5. The American value is the **average of those discounted payoffs** (then $\max$ with the $t=0$ intrinsic).

Paths for pricing are **risk-neutral** (drift $\mu \rightarrow r$ from the option panel; vol/jumps from §4 for that ticker). §5's expected-vs-history plot is visualization only — it is not the input to LSM.

**Workflow:** §4 **Reestimate** → §5 **Start** (optional viz) → §6 **Compute stopping** (all three underlyings).





In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))

from american_lsm import (
    STOP_TICKERS,
    lsm_american_call,
    load_calls,
    params_asof,
    sample_listed_minute_calls,
)

def _show_fig(fig):
    """Show figure once as PNG (same pattern as §5)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _rn_paths_for_contract(row, n_paths: int, seed: int):
    """Risk-neutral paths to expiry using §5 simulator (size-only Q: R ← λR so E[e^{r_t}]=e^{r Δt}, signs unchanged)."""
    ticker = str(getattr(row, "underlying", "SPY")).upper()
    if ticker not in rolling or len(rolling[ticker]) == 0:
        raise RuntimeError(f"No {ticker} calibration — run Reestimate in §4 first.")
    p = params_asof(rolling[ticker], row.trading_date)
    if p is None:
        raise RuntimeError(f"No {ticker} calibration — run Reestimate in §4 first.")
    dte = int(row.dte)
    if dte < 2:
        raise ValueError("dte must be >= 2")
    n_steps = int(getattr(row, "n_steps", 0)) or int(dte)
    r = float(row.r)
    S0 = float(row.S_t)
    steps = {
        "p_uu": np.full(n_steps, float(p["p_uu"]), dtype=float),
        "p_du": np.full(n_steps, float(p["p_du"]), dtype=float),
        "p_ud": np.full(n_steps, float(p["p_ud"]), dtype=float),
        "p_dd": np.full(n_steps, float(p["p_dd"]), dtype=float),
        "p_hh": np.full(n_steps, float(p["p_hh"]), dtype=float),
        "p_ll": np.full(n_steps, float(p["p_ll"]), dtype=float),
        "p_lh": np.full(n_steps, float(p["p_lh"]), dtype=float),
        "p_hl": np.full(n_steps, float(p["p_hl"]), dtype=float),
        "mu_u_l": np.full(n_steps, float(p["mu_u_l"]), dtype=float),
        "sig_u_l": np.full(n_steps, float(p["sig_u_l"]), dtype=float),
        "mu_u_h": np.full(n_steps, float(p["mu_u_h"]), dtype=float),
        "sig_u_h": np.full(n_steps, float(p["sig_u_h"]), dtype=float),
        "mu_d_l": np.full(n_steps, float(p["mu_d_l"]), dtype=float),
        "sig_d_l": np.full(n_steps, float(p["sig_d_l"]), dtype=float),
        "mu_d_h": np.full(n_steps, float(p["mu_d_h"]), dtype=float),
        "sig_d_h": np.full(n_steps, float(p["sig_d_h"]), dtype=float),
        "last_up": np.full(n_steps, float(p["last_up"]), dtype=float),
        "last_wild": np.full(n_steps, float(p["last_wild"]), dtype=float),
    }
    return simulate_modified_gbm_rolling(steps, S0, n_paths, seed, rf=r)


_STOP_TICKERS = list(STOP_TICKERS)
_contracts_by_ticker = {}
for _t in _STOP_TICKERS:
    _c = sample_listed_minute_calls(
        load_calls(DATA, _t, panel="short_interval"),
        prices[_t],
        PERIOD_START,
        PERIOD_END,
        
    )
    if not _c["trading_date"].is_unique:
        raise RuntimeError(f"{_t}: duplicate trading minutes in §6 sample")
    _minute = _c["trading_date"].dt.floor("min")
    if not (_minute == _c["trading_date"]).all():
        raise RuntimeError(f"{_t}: §6 quote times are not on the 1-minute grid")
    _exps = sorted({pd.Timestamp(x).normalize().date() for x in _c["expiration"]})
    print(
        f"{_t}: {len(_c)} contracts | unique minutes={_c['trading_date'].nunique()} "
        f"| listed expiries={_exps} | minute-grid=True"
    )
    _contracts_by_ticker[_t] = _c

stopping_results = {}  # ticker -> DataFrame

for _t in _STOP_TICKERS:
    _n = len(_contracts_by_ticker[_t])
    display(Markdown(
        f"Sampled **{_n}** {_t} American calls in "
        f"{PERIOD_START.date()} → {PERIOD_END.date()} "
        f"(listed expiry; unique 1-minute quote times; percentage RMSE = LSM vs market)."
    ))
    if _n:
        display(
            _contracts_by_ticker[_t][
                ["trading_date", "S_t", "K", "expiration", "dte", "n_steps", "r", "moneyness", "option_price"]
            ].head(8)
        )

_stop_n_paths = widgets.IntSlider(
    value=2000, min=500, max=8000, step=500, description="n_paths",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"),
)
_stop_seed = widgets.IntText(value=42, description="seed", layout=widgets.Layout(width="200px"))
_btn_stop = widgets.Button(
    description="Compute stopping", button_style="primary", icon="calculator"
)
_stop_out = widgets.Output(layout=widgets.Layout(width="100%"))
_stop_busy = {"on": False}


def _run_optimal_stopping(_=None):
    global stopping_results
    if _stop_busy["on"]:
        return
    _stop_busy["on"] = True
    with _stop_out:
        clear_output(wait=True)
        try:
            missing = [t for t in _STOP_TICKERS if t not in rolling or len(rolling[t]) == 0]
            if missing:
                display(Markdown(
                    "Run **Reestimate** in §4 first "
                    f"(need calibration for: {', '.join(missing)})."
                ))
                return

            n_paths = int(_stop_n_paths.value)
            seed0 = int(_stop_seed.value)
            dt = 1.0 / 252
            stopping_results = {}

            for ticker in _STOP_TICKERS:
                contracts = _contracts_by_ticker[ticker]
                if contracts is None or len(contracts) == 0:
                    display(Markdown(f"No {ticker} call contracts in this period panel slice."))
                    continue

                rows = []
                example = None
                for i, row in enumerate(contracts.itertuples(index=False)):
                    paths = _rn_paths_for_contract(row, n_paths, seed0 + i)
                    res = lsm_american_call(paths, K=float(row.K), r=float(row.r), dt=dt)
                    err = res.price - float(row.option_price)
                    rows.append({
                        "ticker": ticker,
                        "trading_date": row.trading_date,
                        "S_t": float(row.S_t),
                        "K": float(row.K),
                        "dte": int(row.dte),
                        "r": float(row.r),
                        "market": float(row.option_price),
                        "model_price": res.price,
                        "error": err,
                        "early_ex_frac": res.early_exercise_frac,
                        "mean_ex_day": res.mean_exercise_step,
                    })
                    if example is None:
                        example = (row, paths, res)

                df = pd.DataFrame(rows)
                stopping_results[ticker] = df
                rmse = float(100.0 * np.sqrt(np.mean((df["error"] / np.maximum(np.abs(df["market"]), 1e-8)) ** 2)))
                mae = float(np.mean(np.abs(df["error"])))
                color = COLORS.get(ticker, "#2ca02c")

                display(Markdown(
                    f"### Modified GBM v2 — LSM results ({ticker})\n"
                    f"n_paths={n_paths} | contracts={len(df)} | "
                    f"RMSE={rmse:.2f}% | MAE={mae:.4f} | "
                    f"mean early-exercise fraction="
                    f"{df['early_ex_frac'].mean():.3f}"
                ))
                display(
                    df[
                        ["trading_date", "S_t", "K", "dte", "market", "model_price",
                         "error", "early_ex_frac", "mean_ex_day"]
                    ].round(4)
                )

                with plt.ioff():
                    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
                    ax = axes[0]
                    ax.scatter(df["market"], df["model_price"], alpha=0.75, color=color)
                    lo = min(df["market"].min(), df["model_price"].min())
                    hi = max(df["market"].max(), df["model_price"].max())
                    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
                    ax.set_xlabel("market option_price")
                    ax.set_ylabel("model LSM price")
                    ax.set_title("Price: model vs market")

                    axes[1].bar(
                        ["model", "market"],
                        [df["model_price"].mean(), df["market"].mean()],
                        color=[color, "#7f7f7f"],
                    )
                    axes[1].set_title("Mean option value")
                    axes[1].set_ylabel("price")

                    axes[2].hist(
                        df["mean_ex_day"], bins=12,
                        color=color, alpha=0.85, edgecolor="white",
                    )
                    axes[2].set_xlabel("mean exercise day (by contract)")
                    axes[2].set_title("Optimal exercise timing")
                    fig.suptitle(
                        f"Modified GBM v2 optimal stopping | {ticker} | "
                        f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                        fontsize=11, y=1.02,
                    )
                    fig.tight_layout()
                _show_fig(fig)

                if example is not None:
                    row, paths, res = example
                    j = int(np.argmin(np.abs(res.exercise_steps - res.mean_exercise_step)))
                    t_ex = int(res.exercise_steps[j])
                    with plt.ioff():
                        fig2, ax = plt.subplots(figsize=(10, 3.8))
                        ax.plot(paths[j], color=color, lw=1.5, label="one RN path")
                        ax.axhline(float(row.K), color="gray", ls="--", lw=1, label=f"K={row.K:g}")
                        ax.scatter(
                            [t_ex], [paths[j, t_ex]], color="crimson", zorder=5, s=50,
                            label=f"exercise day {t_ex}",
                        )
                        ax.set_xlabel("day")
                        ax.set_ylabel("S")
                        ax.set_title(
                            f"{ticker} example path | "
                            f"trade {pd.Timestamp(row.trading_date).date()} | "
                            f"dte={int(row.dte)} | model={res.price:.3f} vs "
                            f"mkt={float(row.option_price):.3f}"
                        )
                        ax.legend(frameon=False, loc="best")
                        fig2.tight_layout()
                    _show_fig(fig2)

            display(Markdown(
                "Results stored in `stopping_results` "
                "(dict keyed by ticker → model_price, error, early_ex_frac, mean_ex_day)."
            ))
        except Exception as exc:
            display(Markdown(f"**Error:** `{type(exc).__name__}: {exc}`"))
        finally:
            _stop_busy["on"] = False


_btn_stop.on_click(_run_optimal_stopping)
display(widgets.VBox([
    widgets.HTML("<b>§6 Optimal stopping — SPY / AAPL / MSFT American calls (LSM)</b>"),
    widgets.HBox([_stop_n_paths, _stop_seed, _btn_stop]),
    _stop_out,
]))





## 7. Reminder

1. **§4:** sliders → **Reestimate** → read parameter tables / rolling charts.
2. **§5:** **Start** → Monte Carlo stock paths + expected vs history (one pair per ticker).
3. **§6:** **Compute stopping** → LSM exercise decision + model vs market on SPY / AAPL / MSFT calls (needs §4).
4. **Restart** (§5) only changes the random seed for path plots.

